# EDA — RACE Reading Comprehension Dataset
**Muhammad Umair (23I-0662) | Abdul Rauf (23I-0591)**

This notebook provides a complete Exploratory Data Analysis of the RACE dataset before any modelling.
We examine:
- Dataset shape and schema
- Missing values and duplicates
- Answer label distribution
- Passage / question / option length distributions
- Vocabulary statistics
- Question-type breakdown
- Cosine similarity distributions
- Sample visualisations

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

OPTION_COLS = ['A', 'B', 'C', 'D']

---
## 1. Load Data

In [ ]:
train_df = pd.read_csv('./data/raw/train.csv')
val_df   = pd.read_csv('./data/raw/val.csv')
test_df  = pd.read_csv('./data/raw/test.csv')

print('Shapes — Train:', train_df.shape, '| Val:', val_df.shape, '| Test:', test_df.shape)
print('\nColumn schema:')
print(train_df.dtypes)

In [ ]:
train_df.head(2)

---
## 2. Missing Values & Duplicates

In [ ]:
def missing_report(df, name):
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    report  = pd.DataFrame({'Missing': missing, 'Pct %': pct})
    print(f'=== {name} ===')
    print(report[report['Missing'] > 0] if report['Missing'].sum() > 0 else '  No missing values ✓')
    print(f'  Duplicate rows: {df.duplicated().sum()}\n')

missing_report(train_df, 'Train')
missing_report(val_df,   'Val')
missing_report(test_df,  'Test')

In [ ]:
# Drop rows with any missing values (consistent with preprocessing.py)
train_df = train_df.dropna().reset_index(drop=True)
val_df   = val_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

# Combine for global statistics
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print('Total records after dropping NaN:', len(all_df))

---
## 3. Answer Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, (df, name) in zip(axes, [(train_df,'Train'), (val_df,'Val'), (test_df,'Test')]):
    counts = df['answer'].value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color=sns.color_palette('muted', 4))
    ax.bar_label(bars, fmt='%d', padding=3)
    ax.set_title(f'{name} — Answer Distribution')
    ax.set_xlabel('Correct Option')
    ax.set_ylabel('Count')
    total = counts.sum()
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                f'{val/total*100:.1f}%', ha='center', va='center',
                color='white', fontweight='bold', fontsize=10)

plt.suptitle('Answer Label Distribution per Split', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\nTrain answer counts:')
print(train_df['answer'].value_counts())
print(f'Balance ratio (should be ~0.25): {(train_df["answer"]=="A").mean():.3f}')

---
## 4. Text Length Distributions

In [ ]:
def word_count(text):
    return len(str(text).split())

def sent_count(text):
    return len([s for s in str(text).split('.') if s.strip()])

# Compute on training set
train_df['article_words']   = train_df['article'].apply(word_count)
train_df['article_sents']   = train_df['article'].apply(sent_count)
train_df['question_words']  = train_df['question'].apply(word_count)
for opt in OPTION_COLS:
    train_df[f'{opt}_words'] = train_df[opt].apply(word_count)

stats_cols = ['article_words', 'article_sents', 'question_words'] + [f'{o}_words' for o in OPTION_COLS]
print('Descriptive statistics — Training set')
print(train_df[stats_cols].describe().round(1).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Article word count
axes[0,0].hist(train_df['article_words'], bins=50, color='steelblue', edgecolor='white')
axes[0,0].axvline(train_df['article_words'].median(), color='red', linestyle='--', label=f'Median={train_df["article_words"].median():.0f}')
axes[0,0].set_title('Article Length (words)')
axes[0,0].set_xlabel('Word Count')
axes[0,0].legend()

# Article sentence count
axes[0,1].hist(train_df['article_sents'], bins=40, color='seagreen', edgecolor='white')
axes[0,1].axvline(train_df['article_sents'].median(), color='red', linestyle='--', label=f'Median={train_df["article_sents"].median():.0f}')
axes[0,1].set_title('Article Length (sentences)')
axes[0,1].set_xlabel('Sentence Count')
axes[0,1].legend()

# Question word count
axes[1,0].hist(train_df['question_words'], bins=30, color='darkorange', edgecolor='white')
axes[1,0].axvline(train_df['question_words'].median(), color='navy', linestyle='--', label=f'Median={train_df["question_words"].median():.0f}')
axes[1,0].set_title('Question Length (words)')
axes[1,0].set_xlabel('Word Count')
axes[1,0].legend()

# Option word count (all four options)
option_lengths = pd.concat([train_df[f'{o}_words'] for o in OPTION_COLS])
axes[1,1].hist(option_lengths, bins=30, color='mediumpurple', edgecolor='white')
axes[1,1].axvline(option_lengths.median(), color='red', linestyle='--', label=f'Median={option_lengths.median():.0f}')
axes[1,1].set_title('Option Length — All A/B/C/D (words)')
axes[1,1].set_xlabel('Word Count')
axes[1,1].legend()

plt.suptitle('Text Length Distributions (Training Set)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Option length comparison across A/B/C/D
opt_data = {opt: train_df[f'{opt}_words'] for opt in OPTION_COLS}
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(opt_data.values(), labels=opt_data.keys(), patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
ax.set_title('Option Word-Count Distribution by Option Letter')
ax.set_ylabel('Word Count')
plt.tight_layout()
plt.show()

---
## 5. Vocabulary Statistics

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Build corpus from training articles
articles_clean = train_df['article'].apply(clean_text)
all_tokens = [tok for text in articles_clean for tok in text.split()]

vocab        = set(all_tokens)
token_freq   = Counter(all_tokens)
total_tokens = len(all_tokens)

print(f'Total tokens (train articles) : {total_tokens:,}')
print(f'Unique vocabulary size        : {len(vocab):,}')
print(f'Type-to-Token Ratio (TTR)     : {len(vocab)/total_tokens:.4f}')
print(f'\nTop-30 most frequent tokens:')
print(pd.DataFrame(token_freq.most_common(30), columns=['Token', 'Count']))

In [ ]:
# Zipf's Law — log-log frequency rank plot
sorted_freqs = sorted(token_freq.values(), reverse=True)
ranks = range(1, len(sorted_freqs) + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(list(ranks)[:20000], sorted_freqs[:20000], color='steelblue', linewidth=1)
ax.set_title("Zipf's Law — Token Frequency vs Rank (log-log)")
ax.set_xlabel('Rank')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Top-30 most frequent content words (excluding stopwords)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

content_freq = Counter({w: c for w, c in token_freq.items() if w not in ENGLISH_STOP_WORDS and len(w) > 2})
top30 = content_freq.most_common(30)

words, counts = zip(*top30)
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(words, counts, color=sns.color_palette('Blues_d', 30))
ax.set_title('Top-30 Content Words in Training Articles')
ax.set_ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 6. Question-Type Analysis

In [ ]:
WH_WORDS = ['what', 'who', 'when', 'where', 'why', 'how', 'which']

def question_type(q: str) -> str:
    q_lower = str(q).lower().strip()
    for wh in WH_WORDS:
        if q_lower.startswith(wh):
            return wh
    if q_lower.startswith('_') or '____' in q_lower or 'blank' in q_lower:
        return 'fill-in-blank'
    return 'other'

train_df['q_type'] = train_df['question'].apply(question_type)
q_type_counts = train_df['q_type'].value_counts()

print('Question type breakdown (train):')
print(q_type_counts.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
q_type_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', len(q_type_counts)))
axes[0].set_title('Question Type Counts')
axes[0].set_xlabel('Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Pie chart
axes[1].pie(q_type_counts.values, labels=q_type_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('muted', len(q_type_counts)))
axes[1].set_title('Question Type Distribution')

plt.suptitle('Question-Type Analysis — Training Set', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Average article length by question type
avg_len = train_df.groupby('q_type')['article_words'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
avg_len.plot(kind='bar', ax=ax, color='teal')
ax.set_title('Avg Article Length by Question Type')
ax.set_ylabel('Mean Word Count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## 7. Correct vs Incorrect Option Similarity

In [ ]:
# Fit a lightweight TF-IDF on a sample for cosine similarity analysis
SAMPLE_N = 3000
sample_df = train_df.sample(SAMPLE_N, random_state=42).reset_index(drop=True)

tfidf_eda = TfidfVectorizer(max_features=5000, stop_words='english', sublinear_tf=True)
all_texts = pd.concat([
    sample_df['article'],
    sample_df['question'],
    *[sample_df[o] for o in OPTION_COLS]
])
tfidf_eda.fit(all_texts.apply(clean_text))

correct_sims, wrong_sims = [], []

for _, row in sample_df.iterrows():
    art_vec = tfidf_eda.transform([clean_text(row['article'])])
    correct = str(row['answer']).strip().upper()
    for opt in OPTION_COLS:
        opt_vec = tfidf_eda.transform([clean_text(row[opt])])
        sim = float(cosine_similarity(art_vec, opt_vec)[0, 0])
        if opt == correct:
            correct_sims.append(sim)
        else:
            wrong_sims.append(sim)

print(f'Correct option cosine sim  — mean: {np.mean(correct_sims):.4f}, std: {np.std(correct_sims):.4f}')
print(f'Incorrect option cosine sim — mean: {np.mean(wrong_sims):.4f}, std: {np.std(wrong_sims):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(correct_sims, bins=40, alpha=0.6, label='Correct option', color='green', edgecolor='white')
ax.hist(wrong_sims,   bins=40, alpha=0.6, label='Incorrect option', color='crimson', edgecolor='white')
ax.axvline(np.mean(correct_sims), color='darkgreen', linestyle='--', linewidth=1.5)
ax.axvline(np.mean(wrong_sims),   color='darkred',   linestyle='--', linewidth=1.5)
ax.set_title('TF-IDF Cosine Similarity: Article ↔ Option\n(Correct vs Incorrect Options)')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Answer Balance Across Splits

In [ ]:
# Side-by-side normalised bar chart across all three splits
splits = {'Train': train_df, 'Val': val_df, 'Test': test_df}

proportions = pd.DataFrame({
    name: df['answer'].value_counts(normalize=True).sort_index() * 100
    for name, df in splits.items()
})

ax = proportions.plot(kind='bar', figsize=(8, 4), colormap='Set2', edgecolor='white')
ax.set_title('Answer Label Proportion (%) Across Splits')
ax.set_xlabel('Correct Option Label')
ax.set_ylabel('Proportion (%)')
ax.set_xticklabels(['A', 'B', 'C', 'D'], rotation=0)
ax.axhline(25, color='black', linestyle=':', linewidth=1, label='Expected 25%')
ax.legend()
plt.tight_layout()
plt.show()

print(proportions.round(2))

---
## 9. Keyword Overlap — Question ↔ Article

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def keyword_overlap(q, art):
    q_toks   = set(clean_text(q).split()) - ENGLISH_STOP_WORDS
    art_toks = set(clean_text(art).split()) - ENGLISH_STOP_WORDS
    if not q_toks:
        return 0.0
    return len(q_toks & art_toks) / len(q_toks)

sample_df['q_art_overlap'] = sample_df.apply(
    lambda r: keyword_overlap(r['question'], r['article']), axis=1
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sample_df['q_art_overlap'], bins=30, color='mediumpurple', edgecolor='white')
ax.axvline(sample_df['q_art_overlap'].mean(), color='red', linestyle='--',
           label=f'Mean={sample_df["q_art_overlap"].mean():.3f}')
ax.set_title('Keyword Overlap Ratio: Question ↔ Article (sample n=3000)')
ax.set_xlabel('Overlap Ratio')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. Pairwise Correlation Between Features

In [ ]:
feature_df = sample_df[['article_words', 'article_sents', 'question_words',
                          'A_words', 'B_words', 'C_words', 'D_words',
                          'q_art_overlap']].copy()

corr = feature_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix (sample)')
plt.tight_layout()
plt.show()

---
## 11. Sample Passages

In [ ]:
def display_sample(df, idx=0):
    row = df.iloc[idx]
    print('='*70)
    print('ARTICLE (first 400 chars):')
    print(row['article'][:400], '...')
    print()
    print('QUESTION :', row['question'])
    print()
    for opt in OPTION_COLS:
        marker = ' ← CORRECT' if opt == str(row['answer']).strip().upper() else ''
        print(f'  {opt}: {row[opt]}{marker}')
    print('='*70)

print('Sample 1 — Training set')
display_sample(train_df, 0)
print()
print('Sample 2 — Training set')
display_sample(train_df, 10)

---
## 12. Summary Statistics Table

In [ ]:
summary = {
    'Split': ['Train', 'Val', 'Test'],
    'Rows':  [len(train_df), len(val_df), len(test_df)],
    'Expanded Rows (×4)': [len(train_df)*4, len(val_df)*4, len(test_df)*4],
    'Avg Article Words':  [
        train_df['article_words'].mean().round(1),
        val_df['article'].apply(word_count).mean().round(1),
        test_df['article'].apply(word_count).mean().round(1),
    ],
    'Avg Q Words': [
        train_df['question_words'].mean().round(1),
        val_df['question'].apply(word_count).mean().round(1),
        test_df['question'].apply(word_count).mean().round(1),
    ],
    'Most Common Answer': [
        train_df['answer'].mode()[0],
        val_df['answer'].mode()[0],
        test_df['answer'].mode()[0],
    ]
}

summary_df = pd.DataFrame(summary)
print('=== RACE Dataset Summary ===')
print(summary_df.to_string(index=False))

---
## 13. Key EDA Findings

| Finding | Implication |
|---|---|
| Answer labels are roughly balanced (~25% each) | No class-weight correction needed for answer prediction; binary verification still needs `class_weight='balanced'` |
| Articles are long (median ~300 words, up to 1000+) | Bag-of-words methods will produce very high-dimensional sparse matrices; `max_features=10000` and sparse storage are essential |
| Questions are short (median ~9 words) | Question text alone is a weak signal; must be combined with article context |
| Correct options have slightly higher cosine similarity to the article than incorrect ones | Cosine similarity is a useful discriminative feature for Model A |
| Vocabulary TTR is low (many repeated tokens) | TF-IDF `sublinear_tf=True` and `min_df=2` will prune rare noisy terms effectively |
| Fill-in-blank questions are the most common type | Template-based generation must handle blank-filling as well as Wh-word questions |